# SQF Optimisation — Interactive Notebook (v2026.1.0)
This notebook is currently being refactored to improve user-friendliness and allow easier reporting.

**Sections:**
- 1. Inputs / dataset
- 2. Model fitting
- 3. Prediction and Score Computation
- 4. Visualisation
- 5. Optimisation and Simulation
- 6. Simulation at arbitrary point

TODO: general cleanup and documentation

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from sqf_optimization.core import fit_models, predict_grid_from_params
from sqf_optimization.io import load_config, load_measurements
from sqf_optimization.scores import compute_scores
from sqf_optimization.visualization import plot_score_surface, plot_synthetic_chromatogram

## Inputs / dataset

TODO: Add documentation


In [ ]:
config = load_config("data/instrument-config_5_amlodipine.json")
measurements = load_measurements("data/measurements_5_amlodipine.csv")

display(config.__dict__)
display(measurements)

## Model fitting

TODO: Description of the model


In [ ]:
model_params = fit_models(config, measurements)
display(model_params)

## Prediction and Score Computation

TODO: Include documentation before code  
TODO: Clean up output


In [ ]:
# --- build predictions on the full grid ---
T_vals = np.linspace(*config.T_levels, config.n_T)
tG_vals = np.linspace(*config.tG_levels, config.n_tG)

pred_grid = predict_grid_from_params(
    model_params,
    T_vals=T_vals,
    tG_vals=tG_vals,
    t0_total=float(config.t0_total_min),
)

# --- compute scores ---
scores_grid = compute_scores(
    pred_grid,
    column_dead_time=float(config.t0_col_min),
    width_penalty_coeff=10.0,
)

display(pred_grid)
display(scores_grid)

## Visualisation

TODO: Plot metric surfaces and SQF; export publication-ready figures.


In [ ]:
for score_name in list(scores_grid.data_vars):
    plot_score_surface(scores_grid, str(score_name)).show()

## Optimisation and Simulation


In [ ]:
# ---- Select optimum point ----
sqf = scores_grid["SQF"].transpose("T", "tG")

# index of the global maximum in the 2D array
i_T, i_tG = np.unravel_index(np.nanargmax(sqf.values), sqf.shape)

# corresponding coordinate values
T_max = float(sqf.coords["T"].values[i_T])
tG_max = float(sqf.coords["tG"].values[i_tG])
sqf_max = float(sqf.values[i_T, i_tG])

print(f"Optimum found at T={T_max:.2f} K, tG={tG_max:.2f} min with SQF={sqf_max:.4f}")
display(scores_grid.isel(T=i_T, tG=i_tG))

In [ ]:
plot_synthetic_chromatogram(pred_grid, i_T, i_tG, measurements).show()


## Simulation at arbitrary point

TODO: Evaluate arbitrary (T, tG), print breakdown, and generate a synthetic chromatogram.


In [ ]:
# ---- Choose a test point (edit as desired) ----
T_test = float(config.T1)   # or any temperature in your domain
tG_test = float(config.tG1) # or any gradient time in your domain

# ---- Predict ----
pred_sp = predict_grid_from_params(
    model_params,
    T_vals=np.array([T_test], dtype=float),
    tG_vals=np.array([tG_test], dtype=float),
    t0_total=float(config.t0_total_min),
)

# ---- Compute scores ----
scores_sp = compute_scores(
    pred_sp,
    column_dead_time=float(config.t0_col_min),
    width_penalty_coeff=10.0,  # keep consistent with your notebook
)

# ---- Display results ----
display(scores_sp)

In [ ]:
# --- Single-point synthetic chromatogram (transcribed to the xarray single-point test) ---

# Assumes you already ran the single-point test cell and have:
# - pred_sp : xr.Dataset with vars tR, w, S and dims (analyte, T, tG)
# - T_test, tG_test : the selected single-point conditions

# Extract per-analyte scalars at the single point (T=0, tG=0)
tR_sp = pred_sp["tR"].isel(T=0, tG=0).values  # (N,)
w_sp  = pred_sp["w"].isel(T=0, tG=0).values   # (N,)
A_sp  = pred_sp["A"].isel(T=0, tG=0).values   # (N,)
analytes = pred_sp.coords["analyte"].values.tolist()

areas = np.array(measurements["area"], dtype=float)  # (N,)

# Convert (w, S) into asymmetric left/right widths (same as old notebook)
w_left  = 2.0 * w_sp / (A_sp + 1.0)
w_right = 2.0 * w_sp * A_sp / (A_sp + 1.0)

# Time axis for simulated chromatogram
x = np.linspace(0.0, float(tG_test), 2000)  # increase points for smoothness

# Broadcasted peak model:
# y_i(x) = a_i * exp( -4 * (tR_i - x)^2 / w_i(x)^2 )
# where w_i(x) = w_left_i if x < tR_i else w_right_i
rt = tR_sp[:, None]            # (N,1)
xx = x[None, :]                # (1,X)
wl = w_left[:, None]           # (N,1)
wr = w_right[:, None]          # (N,1)
aa = areas[:, None]             # (N,1)

w_piece = np.where(xx < rt, wl, wr)  # (N,X)
y = aa * np.exp(-4.0 * ((rt - xx) ** 2) / (w_piece ** 2))  # (N,X)
y_total = np.sum(y, axis=0)  # (X,)


# Overlay individual peaks (comment out if too cluttered)
plt.figure(figsize=(15, 6))
plt.plot(x, y_total, linewidth=2.0, label="Total", color="black")
for i, k in enumerate(analytes):
    plt.plot(x, y[i], label=k, alpha=0.6)
plt.title(f"Predicted chromatogram at tG={tG_test:.2f} min, T={T_test:.2f} K")
plt.xlabel("Time [min]")
plt.ylabel("Signal (a.u.)")
plt.legend(ncol=3, fontsize=8)
plt.tight_layout()
plt.show()

# Simple total chromatogram plot (replace the above if too cluttered)
# plt.figure(figsize=(15, 6))
# plt.plot(x, y_total)
# plt.title(f"Predicted chromatogram at tG={tG_test:.2f} min, T={T_test:.2f} K")
# plt.xlabel("Time [min]")
# plt.ylabel("Signal (a.u.)")
# plt.tight_layout()
# plt.show()


## End of Document

Anything below this line is currently being refactored

---
